# 06 Multivariable Analysis — Reference Solutions

Use the Pine and Cypress Nursing Home Legionnaires' disease line list to practice Modified Poisson regression (adjusted RR)
and logistic regression (adjusted OR), and compare the two.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings

# -- CJK font setup (avoid Chinese labels rendering as boxes) --
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# --- Load the data ---
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
fs_map = {"bedridden": 0, "assisted": 1, "independent": 2}
df["functional_score"] = df["functional_status"].map(fs_map)

## Question 1: Predicting death — Crude RR vs Crude OR

1. Build the `dead` column
2. Compute the case fatality rate
3. Compute the crude RR (Modified Poisson) and crude OR (logistic) at the same time
4. Organize into a comparison table

In [ ]:
# --- Build the outcome variable ---
df["dead"] = (df["outcome"] == "dead").astype(int)

# Convert severity to numbers (non-infected set to 0)
sev_map = {"not_ill": 0, "asymptomatic": 0, "mild": 1, "moderate": 2, "severe": 3}
df["severity_score"] = df["clinical_severity"].map(sev_map)

# Use only the infected for death prediction (non-infected won't die of this disease)
cases = df[df["infected"] == 1].copy()
cfr = cases["dead"].mean()
print(f"Infected: {len(cases)} people, deaths: {cases['dead'].sum()} people")
print(f"Case fatality rate (CFR): {cfr:.1%}")
print(f"→ CFR = {cfr:.1%}, much lower than the 43% attack rate")
print(f"→ We expect the gap between OR and RR to be smaller than when predicting infection\n")

# --- Compute crude RR and crude OR at the same time ---
factors_death = ["age", "comorbidity_chf", "comorbidity_copd",
                 "immunosuppressed", "severity_score"]

crude_rows = []
for var in factors_death:
    # Modified Poisson → crude RR
    poisson = smf.glm(
        f"dead ~ {var}", data=cases,
        family=sm.families.Poisson()
    ).fit(cov_type="HC0", disp=0)
    rr = np.exp(poisson.params[var])
    rr_ci = np.exp(poisson.conf_int().loc[var])

    # Logistic → crude OR
    logit = smf.logit(f"dead ~ {var}", data=cases).fit(disp=0)
    or_val = np.exp(logit.params[var])
    or_ci = np.exp(logit.conf_int().loc[var])

    crude_rows.append({
        "variable": var,
        "crude_RR": round(rr, 3),
        "RR 95% CI": f"{rr_ci[0]:.3f}–{rr_ci[1]:.3f}",
        "crude_OR": round(or_val, 3),
        "OR 95% CI": f"{or_ci[0]:.3f}–{or_ci[1]:.3f}",
    })

crude_df = pd.DataFrame(crude_rows)
print("=== Death prediction: Crude RR vs Crude OR ===")
print(crude_df.to_string(index=False))
print("\n→ With a lower fatality rate (~16%), the gap between OR and RR is much smaller than for infection prediction (43%)")

## Question 2: Multivariable Adjusted RR + Adjusted OR

Build a multivariable model predicting death, using both Modified Poisson and Logistic Regression,
and compare the adjusted RR and adjusted OR side by side.

In [ ]:
# --- Shared formula ---
formula_death = (
    "dead ~ age + comorbidity_chf + comorbidity_copd + "
    "immunosuppressed + severity_score"
)

# --- Modified Poisson → Adjusted RR ---
poisson_multi = smf.glm(
    formula_death, data=cases,
    family=sm.families.Poisson()
).fit(cov_type="HC0", disp=0)

# --- Logistic → Adjusted OR ---
logit_multi = smf.logit(formula_death, data=cases).fit(disp=0, method="lbfgs")

# --- Side-by-side comparison table ---
compare_rows = []
for var in poisson_multi.params.index:
    if var == "Intercept":
        continue
    # Adjusted RR
    rr = np.exp(poisson_multi.params[var])
    rr_ci = np.exp(poisson_multi.conf_int().loc[var])
    # Adjusted OR
    or_val = np.exp(logit_multi.params[var])
    or_ci = np.exp(logit_multi.conf_int().loc[var])
    # How much the OR overestimates relative to the RR
    pct_diff = (or_val - rr) / rr * 100

    compare_rows.append({
        "variable": var,
        "adj_RR": round(rr, 3),
        "RR 95% CI": f"{rr_ci[0]:.3f}–{rr_ci[1]:.3f}",
        "adj_OR": round(or_val, 3),
        "OR 95% CI": f"{or_ci[0]:.3f}–{or_ci[1]:.3f}",
        "OR overest.%": f"{pct_diff:+.1f}%",
    })

compare_df = pd.DataFrame(compare_rows)
print("=== Death prediction: Adjusted RR vs Adjusted OR ===")
print(compare_df.to_string(index=False))

# --- Crude vs Adjusted comparison ---
print("\n=== Crude → Adjusted change (using RR) ===")
for var in factors_death:
    c_row = crude_df[crude_df["variable"] == var].iloc[0]
    a_row = compare_df[compare_df["variable"] == var]
    if len(a_row) == 0:
        continue
    a_row = a_row.iloc[0]
    change = (a_row["adj_RR"] - c_row["crude_RR"]) / c_row["crude_RR"] * 100
    print(f"  {var:25s}  crude_RR={c_row['crude_RR']:.3f}  "
          f"adj_RR={a_row['adj_RR']:.3f}  ({change:+.1f}%)")
print("\n→ The variable with the largest change = the factor most confounded by the others")

## Question 3 (challenge): Model comparison + Forest Plot

1. Build two Modified Poisson models (reduced vs full)
2. Compare the AIC
3. Draw an Adjusted RR forest plot using the better model

In [ ]:
# --- Model A (reduced): 3 predictors ---
model_a = smf.glm(
    "dead ~ age + immunosuppressed + severity_score",
    data=cases, family=sm.families.Poisson()
).fit(cov_type="HC0", disp=0)

# --- Model B (full): 5 predictors ---
model_b = smf.glm(
    "dead ~ age + comorbidity_chf + comorbidity_copd + "
    "immunosuppressed + severity_score",
    data=cases, family=sm.families.Poisson()
).fit(cov_type="HC0", disp=0)

# --- AIC comparison ---
print("=== Model comparison (Modified Poisson) ===")
print(f"  Model A (3 variables) AIC = {model_a.aic:.1f}")
print(f"  Model B (5 variables) AIC = {model_b.aic:.1f}")

best = model_a if model_a.aic < model_b.aic else model_b
best_name = "A" if model_a.aic < model_b.aic else "B"
print(f"  → Model {best_name} is better (smaller AIC = the best balance of explanatory power and parsimony)")

In [ ]:
# --- Forest Plot: Adjusted RR (using the better model) ---
forest_data = []
for var in best.params.index:
    if var == "Intercept":
        continue
    rr = np.exp(best.params[var])
    ci = np.exp(best.conf_int().loc[var])
    forest_data.append({
        "variable": var,
        "RR": rr,
        "ci_lo": ci[0],
        "ci_hi": ci[1],
    })

fdf = pd.DataFrame(forest_data)

fig, ax = plt.subplots(figsize=(8, 4))
y_pos = range(len(fdf))

# Point estimate + confidence interval
ax.errorbar(
    fdf["RR"], y_pos,
    xerr=[fdf["RR"] - fdf["ci_lo"], fdf["ci_hi"] - fdf["RR"]],
    fmt="o", color="#D97757", capsize=4, markersize=8,
    ecolor="#6A9BCC", elinewidth=2,
)

# RR = 1 reference line (no effect)
ax.axvline(x=1, color="gray", linestyle="--", alpha=0.5, label="RR = 1")

ax.set_yticks(list(y_pos))
ax.set_yticklabels(fdf["variable"])
ax.set_xlabel("Adjusted Risk Ratio (RR)")
ax.set_title(f"Death prediction model {best_name} — Adjusted RR forest plot")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# --- Interpretation ---
print("\n=== Independent predictors (RR > 1 and CI does not include 1) ===")
for _, row in fdf.iterrows():
    sig = "✓ significant" if row["ci_lo"] > 1 else "  not significant"
    print(f"  {row['variable']:25s}  RR={row['RR']:.3f}  "
          f"({row['ci_lo']:.3f}–{row['ci_hi']:.3f})  {sig}")

### Interpretation

- **severity_score**: clinical severity is the strongest predictor of death (largest RR), which matches intuition
- **immunosuppressed**: after controlling for severity, immunosuppression may still be an independent risk factor
- **age**: the RR per one-year increase in age looks close to 1, but the cumulative effect is large (e.g., 80 vs 70 is a 10-year difference)
- **RR vs OR**: with a ~16% fatality rate, the gap between OR and RR is smaller than for infection prediction (43% attack rate), confirming the principle "the lower the prevalence, the closer the OR is to the RR"
- **Model selection**: the model with the smaller AIC won't necessarily have every variable significant, but it has a better overall balance
- **Limitation**: there are only ~19 deaths, so the model has limited degrees of freedom and shouldn't include too many variables

## Question 4 Solution

In [ ]:
# --- Data: TB contact investigation (synthetic data, same as the exercise) ---
rng = np.random.default_rng(406)
n = 600

age = rng.normal(45, 15, n).clip(5, 90)
diabetes = (rng.uniform(0, 1, n) < (0.05 + age / 300)).astype(int)
close_contact = rng.binomial(1, 0.4, n)
underweight = rng.binomial(1, 0.15, n)
smoking = rng.binomial(1, 0.25, n)

logit_p = (
    -4.3
    + 0.03 * age
    + 0.8 * diabetes
    + 1.3 * close_contact
    + 0.9 * underweight
    + 0.5 * smoking
)
p_tb = 1 / (1 + np.exp(-logit_p))
active_tb = rng.binomial(1, p_tb)

tb = pd.DataFrame({
    "age": age.round(1),
    "diabetes": diabetes,
    "close_contact": close_contact,
    "underweight": underweight,
    "smoking": smoking,
    "active_tb": active_tb,
})

prevalence = tb["active_tb"].mean()
print(f"Sample size: {len(tb)}, active TB: {tb['active_tb'].sum()} people")
print(f"Prevalence: {prevalence:.1%}\n")

# --- Crude OR: close_contact ---
crude_fit = smf.logit("active_tb ~ close_contact", data=tb).fit(disp=0)
crude_or = np.exp(crude_fit.params["close_contact"])
crude_ci = np.exp(crude_fit.conf_int().loc["close_contact"])
print(f"Crude OR (close_contact) = {crude_or:.2f}  "
      f"(95% CI: {crude_ci[0]:.2f}–{crude_ci[1]:.2f})")

# --- Multivariable model: Adjusted OR ---
multi_fit = smf.logit(
    "active_tb ~ age + diabetes + close_contact + underweight + smoking",
    data=tb,
).fit(disp=0)

rows = []
for var in multi_fit.params.index:
    if var == "Intercept":
        continue
    or_val = np.exp(multi_fit.params[var])
    ci = np.exp(multi_fit.conf_int().loc[var])
    rows.append({
        "variable": var,
        "adj_OR": round(or_val, 3),
        "95% CI": f"{ci[0]:.3f}–{ci[1]:.3f}",
        "significant": "Yes" if (ci[0] > 1 or ci[1] < 1) else "No",
    })
tb_result = pd.DataFrame(rows)
print("\n=== Multivariable model: Adjusted OR ===")
print(tb_result.to_string(index=False))

adj_or_contact = np.exp(multi_fit.params["close_contact"])
change_pct = (adj_or_contact - crude_or) / crude_or * 100
print(f"\ncrude OR = {crude_or:.2f} -> adjusted OR = {adj_or_contact:.2f}  ({change_pct:+.1f}%)")
if abs(change_pct) > 10:
    print("→ The effect of close_contact changed noticeably after adjustment, suggesting the presence of confounders (e.g., age, diabetes)")
else:
    print("→ The effect of close_contact changed little before and after adjustment, so confounding appears minimal")

print("\n=== Risk factors that remain statistically significant after adjustment (95% CI excludes 1) ===")
sig = tb_result[tb_result["significant"] == "Yes"]
print(sig.to_string(index=False))
print(f"\n→ Risk factor with the largest adjusted OR: {tb_result.loc[tb_result['adj_OR'].idxmax(), 'variable']}")

## Question 5 Solution

In [ ]:
# --- Data: COVID-19 confirmed cases from community screening (synthetic data, same as the exercise) ---
rng = np.random.default_rng(507)
n = 600

age = rng.normal(50, 18, n).clip(18, 95)
obesity = rng.binomial(1, 0.30, n)
unvaccinated = rng.binomial(1, 0.35, n)
chronic_lung = rng.binomial(1, 0.12, n)

logit_p = (
    -5.3
    + 0.05 * age
    + 0.8 * obesity
    + 1.1 * unvaccinated
    + 0.9 * chronic_lung
)
p_severe = 1 / (1 + np.exp(-logit_p))
severe = rng.binomial(1, p_severe)

covid = pd.DataFrame({
    "age": age.round(1),
    "obesity": obesity,
    "unvaccinated": unvaccinated,
    "chronic_lung": chronic_lung,
    "severe": severe,
})

severe_rate = covid["severe"].mean()
print(f"Sample size: {len(covid)}, severe cases: {covid['severe'].sum()}")
print(f"Proportion severe: {severe_rate:.1%}\n")

# --- Crude OR: unvaccinated ---
crude_fit = smf.logit("severe ~ unvaccinated", data=covid).fit(disp=0)
crude_or = np.exp(crude_fit.params["unvaccinated"])
crude_ci = np.exp(crude_fit.conf_int().loc["unvaccinated"])
print(f"Crude OR (unvaccinated) = {crude_or:.2f}  "
      f"(95% CI: {crude_ci[0]:.2f}–{crude_ci[1]:.2f})")

# --- Multivariable model: Adjusted OR ---
multi_fit = smf.logit(
    "severe ~ age + obesity + unvaccinated + chronic_lung", data=covid
).fit(disp=0)

rows = []
for var in multi_fit.params.index:
    if var == "Intercept":
        continue
    or_val = np.exp(multi_fit.params[var])
    ci = np.exp(multi_fit.conf_int().loc[var])
    rows.append({
        "variable": var,
        "adj_OR": round(or_val, 3),
        "95% CI": f"{ci[0]:.3f}–{ci[1]:.3f}",
    })
covid_result = pd.DataFrame(rows)
print("\n=== Multivariable model: Adjusted OR ===")
print(covid_result.to_string(index=False))

adj_or_unvax = np.exp(multi_fit.params["unvaccinated"])
adj_ci_unvax = np.exp(multi_fit.conf_int().loc["unvaccinated"])
print(f"\ncrude OR = {crude_or:.2f} -> adjusted OR = {adj_or_unvax:.2f}  "
      f"(95% CI: {adj_ci_unvax[0]:.2f}–{adj_ci_unvax[1]:.2f})")

print("\n=== Interpretation ===")
if adj_ci_unvax[0] > 1:
    print("→ The adjusted OR for unvaccinated is significantly greater than 1 (95% CI excludes 1),")
    print("  meaning that after adjusting for age, obesity, and chronic lung disease, the odds of severe disease among")
    print(f"  unvaccinated people are still {adj_or_unvax:.1f} times those of vaccinated people, supporting the conclusion that vaccination lowers the risk of severe disease")
else:
    print("→ The 95% CI for unvaccinated includes 1; this sample did not confirm a significant association")

## Question 6 Solution

In [ ]:
# --- Data: confirmed dengue cases (synthetic data, same as the exercise) ---
rng = np.random.default_rng(608)
n = 550

secondary_infection = rng.binomial(1, 0.30, n)
age = rng.normal(35, 20, n).clip(1, 85)
diabetes = rng.binomial(1, 0.15, n)
hypertension = rng.binomial(1, 0.20, n)

logit_p = (
    -3.8
    + 1.5 * secondary_infection
    + 0.03 * age
    + 0.7 * diabetes
    + 0.5 * hypertension
)
p_severe = 1 / (1 + np.exp(-logit_p))
severe_dengue = rng.binomial(1, p_severe)

dengue = pd.DataFrame({
    "secondary_infection": secondary_infection,
    "age": age.round(1),
    "diabetes": diabetes,
    "hypertension": hypertension,
    "severe_dengue": severe_dengue,
})

severe_rate = dengue["severe_dengue"].mean()
print(f"Sample size: {len(dengue)}, severe dengue: {dengue['severe_dengue'].sum()} people")
print(f"Proportion severe: {severe_rate:.1%}\n")

# --- Crude OR: secondary_infection ---
crude_fit = smf.logit("severe_dengue ~ secondary_infection", data=dengue).fit(disp=0)
crude_or = np.exp(crude_fit.params["secondary_infection"])
crude_ci = np.exp(crude_fit.conf_int().loc["secondary_infection"])
print(f"Crude OR (secondary_infection) = {crude_or:.2f}  "
      f"(95% CI: {crude_ci[0]:.2f}–{crude_ci[1]:.2f})")

# --- Multivariable model: Adjusted OR ---
multi_fit = smf.logit(
    "severe_dengue ~ secondary_infection + age + diabetes + hypertension", data=dengue
).fit(disp=0)

rows = []
for var in multi_fit.params.index:
    if var == "Intercept":
        continue
    or_val = np.exp(multi_fit.params[var])
    ci = np.exp(multi_fit.conf_int().loc[var])
    rows.append({
        "variable": var,
        "adj_OR": round(or_val, 3),
        "95% CI": f"{ci[0]:.3f}–{ci[1]:.3f}",
    })
dengue_result = pd.DataFrame(rows)
print("\n=== Multivariable model: Adjusted OR ===")
print(dengue_result.to_string(index=False))

adj_or_2nd = np.exp(multi_fit.params["secondary_infection"])
adj_ci_2nd = np.exp(multi_fit.conf_int().loc["secondary_infection"])
change_pct = (adj_or_2nd - crude_or) / crude_or * 100
print(f"\ncrude OR = {crude_or:.2f} -> adjusted OR = {adj_or_2nd:.2f}  "
      f"(95% CI: {adj_ci_2nd[0]:.2f}–{adj_ci_2nd[1]:.2f})  ({change_pct:+.1f}%)")

print("\n=== Interpretation ===")
print(f"→ After adjusting for age, diabetes, and hypertension, the odds of severe dengue for secondary-infection cases")
print(f"  are {adj_or_2nd:.1f} times those of primary-infection cases")
print("→ This result is consistent with the pathological mechanism of ADE (antibody-dependent enhancement:")
print("  non-neutralizing antibodies against the first serotype instead promote entry and replication of the second serotype)"
      if adj_or_2nd > 1 else "→ This sample did not show an enhancement effect from secondary infection")

## Question 7 Solution

In [ ]:
# --- Data: confirmed influenza cases (synthetic data, same as the exercise) ---
rng = np.random.default_rng(709)
n = 700

age = rng.normal(40, 20, n).clip(0, 95)
chronic_disease = rng.binomial(1, 0.22, n)
p_vaccinated = 0.20 + 0.65 * chronic_disease
vaccinated = rng.binomial(1, p_vaccinated)
late_treatment = rng.binomial(1, 0.40, n)

logit_p = (
    -4.1
    + 0.03 * age
    + 1.8 * chronic_disease
    - 0.65 * vaccinated
    + 0.9 * late_treatment
)
p_hosp = 1 / (1 + np.exp(-logit_p))
hospitalized = rng.binomial(1, p_hosp)

flu = pd.DataFrame({
    "age": age.round(1),
    "chronic_disease": chronic_disease,
    "vaccinated": vaccinated,
    "late_treatment": late_treatment,
    "hospitalized": hospitalized,
})

hosp_rate = flu["hospitalized"].mean()
print(f"Sample size: {len(flu)}, hospitalized: {flu['hospitalized'].sum()}")
print(f"Proportion hospitalized: {hosp_rate:.1%}\n")

# --- Crude OR: vaccinated ---
crude_fit = smf.logit("hospitalized ~ vaccinated", data=flu).fit(disp=0)
crude_or = np.exp(crude_fit.params["vaccinated"])
crude_ci = np.exp(crude_fit.conf_int().loc["vaccinated"])
print(f"Crude OR (vaccinated) = {crude_or:.2f}  "
      f"(95% CI: {crude_ci[0]:.2f}–{crude_ci[1]:.2f})")

# --- Multivariable model: Adjusted OR ---
multi_fit = smf.logit(
    "hospitalized ~ age + chronic_disease + vaccinated + late_treatment", data=flu
).fit(disp=0)

rows = []
for var in multi_fit.params.index:
    if var == "Intercept":
        continue
    or_val = np.exp(multi_fit.params[var])
    ci = np.exp(multi_fit.conf_int().loc[var])
    rows.append({
        "variable": var,
        "adj_OR": round(or_val, 3),
        "95% CI": f"{ci[0]:.3f}–{ci[1]:.3f}",
    })
flu_result = pd.DataFrame(rows)
print("\n=== Multivariable model: Adjusted OR ===")
print(flu_result.to_string(index=False))

adj_or_vax = np.exp(multi_fit.params["vaccinated"])
adj_ci_vax = np.exp(multi_fit.conf_int().loc["vaccinated"])
print(f"\ncrude OR (vaccinated) = {crude_or:.2f}  ->  adjusted OR = {adj_or_vax:.2f}  "
      f"(95% CI: {adj_ci_vax[0]:.2f}–{adj_ci_vax[1]:.2f})")

print("\n=== Interpretation ===")
if crude_or >= 1 and adj_or_vax < 1:
    print("→ The crude OR suggests vaccinated people have a higher (or near-1) risk of hospitalization,")
    print("  but after adjusting for chronic disease history, the adjusted OR flips to a protective effect (OR < 1)")
else:
    print("→ The crude OR and adjusted OR point in the same direction, but the gap between them can still be compared")
print("→ This phenomenon is called confounding by indication:")
print("  patients with chronic disease are preferentially recommended for vaccination because they are already higher risk,")
print("  which makes vaccinated positively correlated with chronic_disease; without adjusting for chronic_disease,")
print("  the crude OR would underestimate (or even reverse the sign of) the vaccine's true protective effect")

## Question 8 Solution

In [ ]:
# --- Data: measles outbreak cases (synthetic data, same as the exercise) ---
rng = np.random.default_rng(810)
n = 650

age = rng.uniform(0.5, 15, n)
malnutrition = rng.binomial(1, 0.25, n)
vitamin_a_deficiency = rng.binomial(1, 0.20, n)

logit_p = (
    -2.8
    + 0.6 * malnutrition
    + 0.6 * vitamin_a_deficiency
    + 1.6 * malnutrition * vitamin_a_deficiency
    - 0.05 * age
)
p_severe = 1 / (1 + np.exp(-logit_p))
severe_complication = rng.binomial(1, p_severe)

measles = pd.DataFrame({
    "age": age.round(2),
    "malnutrition": malnutrition,
    "vitamin_a_deficiency": vitamin_a_deficiency,
    "severe_complication": severe_complication,
})


# --- Group comparison (looking for a synergistic effect) ---
def group_label(row):
    if row["malnutrition"] and row["vitamin_a_deficiency"]:
        return "Both"
    if row["malnutrition"]:
        return "Malnutrition only"
    if row["vitamin_a_deficiency"]:
        return "Vitamin A deficiency only"
    return "Neither"


measles["group"] = measles.apply(group_label, axis=1)
group_summary = measles.groupby("group")["severe_complication"].agg(["mean", "count"])
group_summary = group_summary.rename(columns={"mean": "severe_rate", "count": "n"})
print("=== Proportion of severe complications by group ===")
print(group_summary.to_string())
print("\n→ If the proportion for 'Both' is far higher than the sum of the individual effects of")
print("  'Malnutrition only' and 'Vitamin A deficiency only', this suggests the two factors may have a synergistic (supra-multiplicative) effect on risk\n")

# --- Main-effects model (no interaction) ---
main_fit = smf.logit(
    "severe_complication ~ malnutrition + vitamin_a_deficiency + age", data=measles
).fit(disp=0)

print("=== Main-effects model Adjusted OR ===")
for var in ["malnutrition", "vitamin_a_deficiency", "age"]:
    or_val = np.exp(main_fit.params[var])
    ci = np.exp(main_fit.conf_int().loc[var])
    print(f"  {var:22s}  OR={or_val:.3f}  (95% CI: {ci[0]:.3f}–{ci[1]:.3f})")
print(f"  AIC = {main_fit.aic:.1f}\n")

# --- Interaction model ---
inter_fit = smf.logit(
    "severe_complication ~ malnutrition * vitamin_a_deficiency + age", data=measles
).fit(disp=0)

print("=== Interaction model Adjusted OR ===")
for var in inter_fit.params.index:
    if var == "Intercept":
        continue
    or_val = np.exp(inter_fit.params[var])
    ci = np.exp(inter_fit.conf_int().loc[var])
    print(f"  {var:40s}  OR={or_val:.3f}  (95% CI: {ci[0]:.3f}–{ci[1]:.3f})")
print(f"  AIC = {inter_fit.aic:.1f}")

interaction_term = "malnutrition:vitamin_a_deficiency"
inter_or = np.exp(inter_fit.params[interaction_term])
inter_ci = np.exp(inter_fit.conf_int().loc[interaction_term])

print("\n=== Model comparison ===")
print(f"Main-effects model AIC = {main_fit.aic:.1f}")
print(f"Interaction model AIC = {inter_fit.aic:.1f}")
if inter_fit.aic < main_fit.aic:
    print("→ The interaction model has a smaller AIC, so adding the interaction term improved model fit")
else:
    print("→ The interaction model did not clearly improve the AIC; judge this jointly with sample size and CI width")

print("\n=== Interpretation ===")
print(f"Interaction term (malnutrition:vitamin_a_deficiency) OR = {inter_or:.2f}  "
      f"(95% CI: {inter_ci[0]:.2f}–{inter_ci[1]:.2f})")
print("→ This OR represents the extra multiplicative factor from having both risk factors present at once,")
print("  beyond what you'd get by multiplying the two main effects independently (a supra-multiplicative interaction)")
if inter_or > 1:
    print("→ OR > 1 shows that when malnutrition and vitamin A deficiency are both present, the increase in severe-complication risk")
    print("  exceeds the product of their individual effects—the two factors act synergistically in this model")
else:
    print("→ OR <= 1; this sample did not show a synergistic effect")
print("→ Public health implication: when screening measles cases, prioritize identifying children with both malnutrition and vitamin A deficiency,")
print("  and provide early vitamin A supplementation with close follow-up to reduce the risk of severe complications")